# JamCoders Day 3 Lecture 1

### Boaz Barak

Going deeper into functions

In [51]:
def exponent(x,y):
    total = 1
    for i in range(abs(y)):
        if y < 0:
            total /= x
        else:
            total *= x
    return total
exponent(2,-3)

0.125

In [56]:
def multiply(A,B):
    C = []
    for i in range(len(A)+len(B)):
        C = C + [0]
    for i in range(len(A)):
        for j in range(len(B)):
            C[i+j] += A[i]*B[j]
            if C[i+j]>=10:
                C[i+j] = C[i+j]-10
                C[i+j+1] += 1
    
    return C

multiply([2,3],[5])

[0, 6, 1]

## Ignore this section

In [ ]:
def md_table(array):
    """ the same input as above """

    nl = "\n"
    t = 30

    markdown = nl
    markdown += f"| {' | '.join([x.ljust(t-2) for x in array[0]])} |"

    markdown += nl
    markdown += f"| {' | '.join(['-'*(t-2)]*len(array[0]))} |"

    markdown += nl
    for entry in array[1:]:
        L = [entry[0].ljust(t-2)] + [ f'<font color="blue"><b>{x}</b></font>'.ljust(t-2) for x in entry[1:]]
        markdown += f"| {' | '.join(L)} |{nl}"

    return markdown

from IPython.display import Markdown, display
from IPython.display import clear_output

display(Markdown(md_table([["Name","Age"],["Boaz","48"]])))

In [ ]:
MARKDOWN = ""
DEBUG = True
OUTPUT = ""
def print_(*L):
    global OUTPUT
    s = " ".join([str(a) for a in L])
    OUTPUT += s+"\n"

def state(code,line,variables,depth):
    if not DEBUG: return
    lines = code.split('\n')
    lines[line] += " #<-------"     
    code = "\n".join(lines)
    global MARKDOWN
    global DEPTHS
    md_lines = MARKDOWN.split("\n")
    if depth>0:
        found = [i for i,x in enumerate(md_lines) if x[:9]=="```python" ]
        if len(found)>depth:
            md_lines = md_lines[:found[depth]]
        md = "\n".join(md_lines)+"\n"
    else:
        md = ""
    md += rf"""```python
    {code}```
    
    """
    table = [["Variable", "Value"]] + [[k,v] for k,v in variables.items()]
    md += md_table(table) +"\n\n\n"
    MARKDOWN = md
    if OUTPUT:
        MARKDOWN += "\n------------------\n"+f"\n```\n{OUTPUT}\n```"
    clear_output()
    #if depth: print(found, len(md_lines))
    display(Markdown(MARKDOWN))
    if input("---"*depth+":")=="q":
        raise Exception()

def lookinto(name):
    global DEBUG
    DEBUG=True
    global OUTPUT
    OUTPUT = ""
    globals()[name] = globals()[name+"_"]
    
def normal():
    global DEBUG
    DEBUG = False

In [ ]:
code_m ="""
def multiply(a,b):
    result = 0
    for i in range(b):
        result += a
    return result
"""

import inspect

def up_locals():
    frame = inspect.currentframe()
    return frame.f_back.f_back.f_locals
        
        
def multiply_(a,b,depth=0):
    def s(i): 
        state(code_m,i+1,{k:v for k,v in up_locals().items() if k!="s" and k!="depth"},depth)
    result = 0 #1
    s(1)
    for i in range(b): #2
        s(2)
        result += a #3
        s(3)
    s(4)
    return result #4


In [ ]:
code_e = """
def exponent(x,y):
    result = 1
    for i in range(y):
        result = multiply(result,x)
    return result
"""

def exponent_(x,y,depth=0):
    def s(i): 
        state(code_e,i+1,{k:v for k,v in up_locals().items() if k!="s" and k!="depth"},depth)
    result = 1 #1
    s(1)
    for i in range(y): #2
        s(2)
        s(3)
        result = multiply(result,x, depth=1) #3
    s(4)
    return result #4
        

In [ ]:
code_f1 = """
def func1(x):
    j = 10
    print("j=",j, "x=",x)
"""
def func1_(x,depth=0):
    def s(i):
        state(code_f1,i+1,{k:v for k,v in up_locals().items() if k!="s" and k!="depth"},depth)
    j = 10 #1
    s(1)
    print_("j=",j, "x=",x) #2
    s(2)

code_f2 ="""
def func2(x):
    j = 5
    func1(j+x)
    print("x=",x)
"""
def func2_(x, depth=0):
    global OUTPUT
    OUTPUT = ""
    def s(i):
        state(code_f2,i+1,{k:v for k,v in up_locals().items() if k!="s" and k!="depth"},depth)
    j = 5 #1
    s(1)
    s(2)
    func1(j+x, depth=1) #2
    print_("x=",x) #3
    s(3)

## Calling functions example

__Question:__ Write function `multiply` that takes inputs integers $a,b$ and returns $a\times b$ without using `*`

__Hint:__ Use the fact that 
$$a\times b \;=\; \underbrace{b+b+\cdots +b}_{a\; \text{times}}$$

```python
def multiply(a,b):
    # body of function here
    return ...

print(multiply(7,6))
# should give 42
```

In [35]:
# question: multiply two integer without using *
def sign(x):
    if x>=0:
        return +1
    return -1

def abs(x):
    if x>=0:
        return x
    return -x

def multiply(a,b):
    result = 0
    for i in range(abs(b)):
        result = result + sign(b)*a
    return result

In [38]:
print(multiply(6,7), multiply(6,-7))

42 -42


## Diving in 

In [31]:
lookinto("multiply")

In [32]:
print(multiply(3,5))

```python
    
def multiply(a,b):
    result = 0
    for i in range(b):
        result += a
    return result #<-------
```

    
| Variable                     | Value                        |
| ---------------------------- | ---------------------------- |
| a                            | <font color="blue"><b>3</b></font> |
| b                            | <font color="blue"><b>5</b></font> |
| result                       | <font color="blue"><b>15</b></font> |
| i                            | <font color="blue"><b>4</b></font> |





: 


15


In [39]:
normal()

__Question:__ Write a function `exponent` that takes as input two integers $x,y$ and returns $x^y$ without using `*` or `**`. It can use `multiply`

__Hint:__ Use the fact that 
$$x^y = \underbrace{x \times x \times \cdots \times x}_{y\;\text{times}}$$

```python
def exponent(x,y):
    # compute x^y without using * or **
    return ...

print(exponent(2,3))
# should be 8
```

Bonus to think about write `def multiply(A,B)` so that if A and B are lists of digits of the numbers $a$ and $b$ then the return value would be the list of digits of the number $a\times b$

In [40]:
def exponent(x,y):
    result = 1
    for i in range(y):
        result = multiply(result,x)
    return result

In [43]:
print(exponent(2,10))

1024


In [44]:
lookinto("exponent")

In [48]:
lookinto("multiply")
print(exponent(2,3)) 


```python
    
def exponent(x,y):
    result = 1
    for i in range(y):
        result = multiply(result,x)
    return result #<-------
```

    
| Variable                     | Value                        |
| ---------------------------- | ---------------------------- |
| x                            | <font color="blue"><b>2</b></font> |
| y                            | <font color="blue"><b>3</b></font> |
| result                       | <font color="blue"><b>8</b></font> |
| i                            | <font color="blue"><b>2</b></font> |





: 


8


## Reminder - difference between print and return

In [ ]:
def passing_grade(score): # return version
    if score > 50:
        return True
    return False

In [ ]:
if passing_grade(99):
    print("That's a great grade")

In [ ]:
def passing_grade(score): # print version
    if score > 50:
        print("True")
    print("False")

In [ ]:
if passing_grade(99):
    print("That's a great grade")

In [ ]:
def passing_grade(score): # return version
    if score > 50:
        return True
    return False

# Using a function

Suppose we have the `passing_grade` function above. Write a function `passed` that takes as input two lists `students` and `grades` and prints the names of the students that passed.

```python
def passed( students , grades):
    # should be lists of the same length
    # should return a list with the names of students that passed
```

**Example:** `passed( [ "Boaz","Anushka" ] , [ 40, 80 ])` will retuen the list `[ "Anushka" ]`

In [ ]:
def passed(students,grades):
    n = len(students)
    result = []
    for i in range(n):
        if passing_grade(grades[i]):
            result += [ students[i] ] 
    return result

In [ ]:
students = [ "Boaz", "Anushka" ]
grades = [ 40, 80 ]
print(passed(students,grades))

In [ ]:
# let's make grades random for fairness
students = ["Adrianna", "Naila", "Daniel", "Robert", "Kanav", "Anushka", "Andrey", "Mallika", "Sarika", "Frank", "Bezawit"]
import random
grades = []
for i in range(len(students)):
    grades += [ random.randint(1,100) ]
print(grades)

In [ ]:
passed(students,grades)

# Functions (variable scoping)

In [ ]:
# Variables inside a function have no relation to variables outside the function
# See what happens when "i" is printed inside and outside the function

i = 0
def func(x):
    i = 10
    print("i=",i, "x=",x)
    
func(5)
print("i=",i)

In [ ]:
# Variables across functions have no relation to each other
# A variable declared inside a function will have no relation to outside the function
# Why is there an error?

def func1(x):
    j = 10
    print("j=",j, "x=",x)

def func2(x):
    j = 5
    func1(j+x)
    print("x=",x)
    
func2(3)

In [ ]:
lookinto("func2")
lookinto("func1")
func2(3)

In [ ]:
normal()

In [ ]:
# Why is this the output?

k = 0
for i in range(3):
    print(k)